<a href="https://colab.research.google.com/github/TheVailen/frameworks-lab2-time-series/blob/main/lab2_weather_timeseries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Лабораторная работа 2
## Что было обнаружено в исходном файле

При первичном просмотре файла `have_fun.xlsx` обнаружились реальные проблемы исходных данных:

- в таблице 8 листов, но содержательных листов с рядами ровно 3
- два рабочих листа скрыты
- названия листов и городов частично испорчены битой кодировкой
- есть опечатки в названиях городов
- на одном листе есть лишние пустые столбцы
- присутствуют маркеры пропусков: `NAN`, `---`, `?`, `нет данных`;
- часть числовых столбцов смешана с текстовыми категориями:
  - `жарко / нормально / холодно`
  - `сухо / морось / ливень`
  - `солнечно / пасмурно / дождь / снег`
  - `тихо / ветрено / шторм / туман / гроза`
- присутствуют выбросы, например:
  - температура **79°C**



In [43]:
!pip install -q optuna

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openpyxl

from scipy import stats
from scipy.signal import periodogram

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import STL

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error

import optuna

In [ ]:
RANDOM_STATE = 42


HORIZON = 24 * 7
WINDOW = 24 * 7
ORIGIN_STRIDE = 24
N_TRIALS = 12
N_JOBS_RF = -1

TARGET_COL = "temperature_2m"
TIME_COL = "ds"
CITY_COL = "city"

BASE_NUM_COLS = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "snowfall",
    "weathercode",
    "wind_speed_10m",
    "surface_pressure",
]

XLSX_PATH = "have_fun.xlsx"

## 1. Получаем доступ ко всем листам Excel, включая скрытые





In [ ]:
def fix_encoding(text):
    if isinstance(text, str):
        try:
            return text.encode("latin1").decode("cp1251")
        except Exception:
            return text
    return text

wb = openpyxl.load_workbook(XLSX_PATH, data_only=False)

sheet_meta = []
for ws in wb.worksheets:
    sheet_meta.append({
        "sheet_name_raw": ws.title,
        "sheet_name_decoded": fix_encoding(ws.title),
        "state": ws.sheet_state,
        "max_row": ws.max_row,
        "max_col": ws.max_column,
    })

sheet_meta_df = pd.DataFrame(sheet_meta)
sheet_meta_df

## Показываем все листы и выделяем те, где реально есть данные

In [ ]:

display(sheet_meta_df)

data_like_sheets = sheet_meta_df.query("max_row > 10").copy()
display(data_like_sheets)

## Ожидаемо: рабочих листов с данными должно оказаться ровно 3

In [ ]:
assert len(data_like_sheets) == 3, f"Ожидалось 3 рабочих листа, найдено {len(data_like_sheets)}"
print("Найдено 3 рабочих листа.")

## 2. Загружаем только 3 рабочих листа

In [ ]:
data_sheet_names = data_like_sheets["sheet_name_raw"].tolist()

raw_sheets = {}
for s in data_sheet_names:
    df = pd.read_excel(XLSX_PATH, sheet_name=s, engine="openpyxl")
    raw_sheets[s] = df

for s, df in raw_sheets.items():
    print("=" * 100)
    print("RAW SHEET:", s, "->", fix_encoding(s))
    print("shape:", df.shape)
    display(df.head(3))

## 3. Проблемы по каждому листу

In [ ]:
def quick_sheet_audit(df, sheet_name):
    out = {}

    out["sheet_name_raw"] = sheet_name
    out["sheet_name_decoded"] = fix_encoding(sheet_name)
    out["rows"] = len(df)
    out["cols"] = len(df.columns)

    unnamed_cols = [c for c in df.columns if str(c).startswith("Unnamed")]
    out["unnamed_cols_count"] = len(unnamed_cols)
    out["all_null_unnamed_cols"] = sum(df[c].isna().all() for c in unnamed_cols)

    if TIME_COL in df.columns:
        out["ds_na"] = int(df[TIME_COL].isna().sum())
        out["time_min"] = df[TIME_COL].min()
        out["time_max"] = df[TIME_COL].max()
        out["sorted_by_time"] = bool(df[TIME_COL].is_monotonic_increasing)
    else:
        out["ds_na"] = None
        out["time_min"] = None
        out["time_max"] = None
        out["sorted_by_time"] = None

    if CITY_COL in df.columns:
        city_decoded = df[CITY_COL].map(fix_encoding)
        out["unique_city_labels_raw"] = int(city_decoded.nunique(dropna=True))
        out["city_value_counts_top10"] = city_decoded.value_counts(dropna=False).head(10).to_dict()
    else:
        out["unique_city_labels_raw"] = None
        out["city_value_counts_top10"] = None

    non_numeric_info = {}
    for col in df.columns:
        if col in [TIME_COL, CITY_COL]:
            continue
        ser = df[col]
        if ser.dtype == "O":
            text_values = ser.dropna().astype(str)
            bad = text_values[pd.to_numeric(text_values, errors="coerce").isna()]
            if len(bad) > 0:
                non_numeric_info[col] = {
                    "bad_count": int(len(bad)),
                    "examples": [fix_encoding(x) for x in bad.head(10).tolist()]
                }
    out["object_non_numeric"] = non_numeric_info

    return out

audit_rows = [quick_sheet_audit(df, s) for s, df in raw_sheets.items()]
audit_df = pd.DataFrame(audit_rows)
display(audit_df[[
    "sheet_name_decoded", "rows", "cols", "unnamed_cols_count", "all_null_unnamed_cols",
    "ds_na", "time_min", "time_max", "sorted_by_time", "unique_city_labels_raw"
]])

In [ ]:
for row in audit_rows:
    print("=" * 120)
    print("Лист:", row["sheet_name_decoded"])
    print("Top city labels:", row["city_value_counts_top10"])
    print("Нечисловые значения в числовых столбцах:")
    for col, info in row["object_non_numeric"].items():
        print(f"  - {col}: {info['bad_count']} шт.; примеры = {info['examples']}")

### Ключевые выводы

- **Лист1**  
  - перемешан по времени;
  - битая кириллица в `city`;
  - много опечаток в названиях двух городов;
  - есть дубли по времени после нормализации названий;
  - часть числовых значений отсутствует.

- **Лист2 - строковые NaN выбросы**  
  - скрытый лист;
  - есть лишние пустые столбцы;
  - есть пустые строки;
  - строковые обозначения пропусков (`NAN`, `---`, `?`, `нет данных`);
  - выраженные выбросы в температуре и давлении;
  - пропуски времени для одного из городов.

- **Лист3 - смешанные типы**  
  - скрытый лист;
  - смешанные типы в нескольких столбцах;
  - текстовые погодные категории вместо чисел;
  - опечатки в городе `Сочи`;
  - временной индекс полный, но значения требуют унификации.

## 4. Очистка и унификация данных

In [ ]:
import re

# Нормализация города
def standardize_city(x):
    x = fix_encoding(x) if isinstance(x, str) else x
    if pd.isna(x):
        return np.nan

    xx = x.strip().lower()

    mapping = {
        "геленджик": "Геленджик",
        "геленджикк": "Геленджик",
        "геленджи": "Геленджик",
        "геленджик ": "Геленджик",
        "геленджик.": "Геленджик",
        "геленджик,": "Геленджик",
        "геленджик-": "Геленджик",
        "геленджик_": "Геленджик",
        "геленджик\n": "Геленджик",
        "геленджик\r\n": "Геленджик",
        "геленджик\t": "Геленджик",
        "геленджиккк": "Геленджик",
        "геленджикь": "Геленджик",
        "геленджикг": "Геленджик",
        "геленджикж": "Геленджик",
        "геленджикй": "Геленджик",
        "геленджикя": "Геленджик",
        "геленджикю": "Геленджик",
        "геленджикъ": "Геленджик",
        "геленджикы": "Геленджик",
        "геленджикэ": "Геленджик",
        "геленджикьь": "Геленджик",

        "благовещенск": "Благовещенск",
        "благовещенскк": "Благовещенск",
        "благовещенс": "Благовещенск",

        "москва": "Москва",
        "мосва": "Москва",

        "находка": "Находка",

        "санкт-петербург": "Санкт-Петербург",

        "сочи": "Сочи",
        "сычи": "Сочи",
        "счи": "Сочи",
        "соч": "Сочи",
    }

    if xx in mapping:
        return mapping[xx]

    xx2 = re.sub(r"[^а-яa-z\- ]", "", xx)
    if xx2 in mapping:
        return mapping[xx2]

    return x.strip().title()


MISSING_MARKERS = {
    "nan", "NAN", "---", "?", "нет данных", "нет", "none", "null", ""
}

WEATHERCODE_TEXT_MAP = {
    "солнечно": 0.0,
    "пасмурно": 3.0,
    "дождь": 61.0,
    "снег": 71.0,
}

TEMP_TEXT_MAP = {
    "холодно": np.nan,
    "нормально": np.nan,
    "жарко": np.nan,
}

PRECIP_TEXT_MAP = {
    "сухо": 0.0,
    "морось": np.nan,
    "ливень": np.nan,
}

WIND_TEXT_MAP = {
    "тихо": np.nan,
    "ветрено": np.nan,
    "шторм": np.nan,
    "туман": np.nan,
    "гроза": np.nan,
}


def normalize_scalar_value(x, column=None):
    if pd.isna(x):
        return np.nan

    if isinstance(x, str):
        x0 = fix_encoding(x).strip()

        if x0 in MISSING_MARKERS or x0.lower() in {m.lower() for m in MISSING_MARKERS}:
            return np.nan

        if column == "weathercode" and x0.lower() in WEATHERCODE_TEXT_MAP:
            return WEATHERCODE_TEXT_MAP[x0.lower()]

        if column == "temperature_2m" and x0.lower() in TEMP_TEXT_MAP:
            return TEMP_TEXT_MAP[x0.lower()]

        if column == "precipitation" and x0.lower() in PRECIP_TEXT_MAP:
            return PRECIP_TEXT_MAP[x0.lower()]

        if column == "wind_speed_10m" and x0.lower() in WIND_TEXT_MAP:
            return WIND_TEXT_MAP[x0.lower()]

        x1 = x0.replace(",", ".")
        try:
            return float(x1)
        except Exception:
            return np.nan

    return x

In [ ]:
def read_and_standardize_sheet(sheet_name, df):
    df = df.copy()

    # 1) убрать полностью пустые столбцы
    keep_cols = [c for c in df.columns if not df[c].isna().all()]
    df = df[keep_cols].copy()

    # 2) приводим city
    df[CITY_COL] = df[CITY_COL].map(standardize_city)

    # 3) приводим смешанные типы к числам
    for col in [c for c in BASE_NUM_COLS if c in df.columns]:
        df[col] = df[col].apply(lambda x: normalize_scalar_value(x, column=col))
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 4) удаляем строки, где нет времени или города
    df = df.dropna(subset=[TIME_COL, CITY_COL]).copy()

    # 5) убираем точные / эквивалентные дубли по (город, время)
    agg_map = {c: "mean" for c in BASE_NUM_COLS if c in df.columns}
    df = (
        df.groupby([CITY_COL, TIME_COL], as_index=False)
          .agg(agg_map)
    )

    # 6) сортировка по времени внутри города
    df = df.sort_values([CITY_COL, TIME_COL]).reset_index(drop=True)

    return df

cleaned_parts = []
for sheet_name, df in raw_sheets.items():
    part = read_and_standardize_sheet(sheet_name, df)
    part["source_sheet"] = fix_encoding(sheet_name)
    cleaned_parts.append(part)

df_all = pd.concat(cleaned_parts, ignore_index=True)
df_all = df_all.sort_values([CITY_COL, TIME_COL]).reset_index(drop=True)

print(df_all.shape)
display(df_all.head())
display(df_all.tail())

### Проверка состава городов после нормализации

In [ ]:
city_counts = (
    df_all.groupby(CITY_COL)
          .agg(
              rows=(TIME_COL, "size"),
              time_min=(TIME_COL, "min"),
              time_max=(TIME_COL, "max")
          )
          .sort_index()
)
display(city_counts)

In [ ]:
print("Города:", sorted(df_all[CITY_COL].unique()))

## 5. Приведение к единому временному индексу

In [ ]:
global_start = df_all[TIME_COL].min()
global_end = df_all[TIME_COL].max()
full_hourly_index = pd.date_range(global_start, global_end, freq="h")

print("Период:", global_start, "->", global_end)
print("Часов в полном индексе:", len(full_hourly_index))

In [ ]:
cities = sorted(df_all[CITY_COL].unique())

panel_parts = []
for city in cities:
    city_df = df_all[df_all[CITY_COL] == city].copy().set_index(TIME_COL).sort_index()
    city_df = city_df.reindex(full_hourly_index)
    city_df.index.name = TIME_COL
    city_df[CITY_COL] = city
    panel_parts.append(city_df.reset_index())

panel = pd.concat(panel_parts, ignore_index=True)
panel = panel.sort_values([CITY_COL, TIME_COL]).reset_index(drop=True)

print(panel.shape)
display(panel.head())

## Сколько пропусков по времени появилось после reindex

In [ ]:
missing_summary = (
    panel.groupby(CITY_COL)
         .apply(lambda g: g[BASE_NUM_COLS].isna().all(axis=1).sum())
         .rename("full_missing_rows_after_reindex")
         .reset_index()
)
display(missing_summary)

## 6. Обработка пропусков и выбросов

## Физически разумные диапазоны для контроля качества
## Значения вне диапазона считаем выбросами -> NaN -> затем интерполяция

In [ ]:
PHYSICAL_BOUNDS = {
    "temperature_2m": (-60, 60),
    "relative_humidity_2m": (0, 100),
    "precipitation": (0, 500),
    "rain": (0, 500),
    "snowfall": (0, 100),
    "weathercode": (0, 99),
    "wind_speed_10m": (0, 150),
    "surface_pressure": (870, 1100),
}

def clip_outliers_iqr_citywise(df, col, whisker=3.0):
    df = df.copy()
    for city, idx in df.groupby(CITY_COL).groups.items():
        s = df.loc[idx, col]
        s_non_na = s.dropna()
        if len(s_non_na) < 50:
            continue
        q1 = s_non_na.quantile(0.25)
        q3 = s_non_na.quantile(0.75)
        iqr = q3 - q1
        lo = q1 - whisker * iqr
        hi = q3 + whisker * iqr

        # Пересечение с физическими границами
        p_lo, p_hi = PHYSICAL_BOUNDS[col]
        lo = max(lo, p_lo)
        hi = min(hi, p_hi)

        bad_mask = (df.loc[idx, col] < lo) | (df.loc[idx, col] > hi)
        df.loc[idx[bad_mask], col] = np.nan
    return df

panel_clean = panel.copy()

# 1) первичный отсев по физическим диапазонам
for col, (lo, hi) in PHYSICAL_BOUNDS.items():
    if col in panel_clean.columns:
        panel_clean.loc[(panel_clean[col] < lo) | (panel_clean[col] > hi), col] = np.nan

# 2) более мягкая IQR-фильтрация выбросов
for col in BASE_NUM_COLS:
    panel_clean = clip_outliers_iqr_citywise(panel_clean, col, whisker=3.0)

# 3) интерполяция / заполнение пропусков по времени внутри города
def impute_city_block(g):
    g = g.sort_values(TIME_COL).copy().set_index(TIME_COL)

    # сначала линейная интерполяция по времени
    interp_cols = [c for c in BASE_NUM_COLS if c in g.columns]
    g[interp_cols] = g[interp_cols].interpolate(method="time", limit_direction="both")

    # weathercode делаем дискретным кодом
    if "weathercode" in g.columns:
        g["weathercode"] = g["weathercode"].round()

    if "relative_humidity_2m" in g.columns:
        g["relative_humidity_2m"] = g["relative_humidity_2m"].clip(0, 100)

    for c in ["precipitation", "rain", "snowfall", "wind_speed_10m"]:
        if c in g.columns:
            g[c] = g[c].clip(lower=0)

    return g.reset_index()

panel_clean = (
    panel_clean.groupby(CITY_COL, group_keys=False)
               .apply(impute_city_block)
               .reset_index(drop=True)
)

display(panel_clean.head())

In [ ]:
# Контроль качества после очистки
quality_after = pd.DataFrame({
    "missing_total": panel_clean[BASE_NUM_COLS].isna().sum(),
    "min": panel_clean[BASE_NUM_COLS].min(),
    "max": panel_clean[BASE_NUM_COLS].max(),
}).sort_index()

display(quality_after)

## 7. Итоговая аккуратная таблица данных

In [ ]:
final_df = panel_clean.copy()
final_df = final_df[[TIME_COL, CITY_COL] + BASE_NUM_COLS].copy()
final_df = final_df.sort_values([CITY_COL, TIME_COL]).reset_index(drop=True)

print(final_df.shape)
display(final_df.head())
display(final_df.sample(5, random_state=RANDOM_STATE))

## Сохраним восстановленный датафрейм отдельно

In [ ]:
OUT_DIR = Path("artifacts")
OUT_DIR.mkdir(exist_ok=True)

clean_csv_path = OUT_DIR / "weather_panel_clean.csv"
final_df.to_csv(clean_csv_path, index=False)

print("Сохранено:", clean_csv_path.resolve())

## 8. Визуализация рядов

In [ ]:
def plot_city_temperature(df, city, start=None, end=None, figsize=(15, 4)):
    tmp = df[df[CITY_COL] == city].copy()
    if start is not None:
        tmp = tmp[tmp[TIME_COL] >= pd.Timestamp(start)]
    if end is not None:
        tmp = tmp[tmp[TIME_COL] <= pd.Timestamp(end)]

    plt.figure(figsize=figsize)
    plt.plot(tmp[TIME_COL], tmp[TARGET_COL], linewidth=1)
    plt.title(f"{city}: {TARGET_COL}")
    plt.xlabel("Время")
    plt.ylabel("Температура")
    plt.grid(True, alpha=0.3)
    plt.show()

for city in sorted(final_df[CITY_COL].unique())[:3]:
    plot_city_temperature(final_df, city, start="2024-01-01", end="2024-03-01")

## 9. Проверка стационарности

ADF-тест корректнее интерпретировать на более сглаженном ряде.  
Ниже берём **дневное среднее** температуры по каждому городу, чтобы получить более читаемую диагностику.

In [ ]:
daily_temp = (
    final_df.set_index(TIME_COL)
            .groupby(CITY_COL)[TARGET_COL]
            .resample("D")
            .mean()
            .reset_index()
)

adf_rows = []
for city, g in daily_temp.groupby(CITY_COL):
    s = g[TARGET_COL].dropna()
    stat, pvalue, lags, nobs, crit, icbest = adfuller(s, autolag="AIC")
    adf_rows.append({
        "city": city,
        "adf_stat": stat,
        "pvalue": pvalue,
        "nobs": nobs,
        "stationary_at_5pct": pvalue < 0.05
    })

adf_df = pd.DataFrame(adf_rows).sort_values("pvalue")
display(adf_df)

## Вывод:
По результатам ADF-теста для всех городов получены `p-value < 0.05`, поэтому гипотеза о нестационарности отвергается. Это означает, что дневные средние температуры в целом ведут себя как стационарные ряды. При этом вывод относится именно к сглаженному дневному ряду, а не к исходным почасовым данным

Обычно для погодных рядов чистая стационарность не выполняется: есть выраженная сезонность по году и по суткам

## 10. Декомпозиция: тренд, сезонность, остатки

In [ ]:
city_for_decomp = "Москва" if "Москва" in daily_temp[CITY_COL].unique() else daily_temp[CITY_COL].iloc[0]
s = (
    daily_temp[daily_temp[CITY_COL] == city_for_decomp]
    .set_index(TIME_COL)[TARGET_COL]
    .asfreq("D")
    .interpolate()
)

stl = STL(s, period=365, robust=True)
res = stl.fit()

fig = res.plot()
fig.set_size_inches(14, 8)
fig.suptitle(f"STL-декомпозиция дневной температуры: {city_for_decomp}", y=1.02)
plt.show()

## 11. Спектральный анализ

In [ ]:
city_for_fft = "Санкт-Петербург" if "Санкт-Петербург" in final_df[CITY_COL].unique() else final_df[CITY_COL].iloc[0]
s_hourly = (
    final_df[final_df[CITY_COL] == city_for_fft]
    .set_index(TIME_COL)[TARGET_COL]
    .asfreq("h")
    .interpolate()
)

freqs, power = periodogram(s_hourly.values, fs=1.0)
mask = freqs > 0

plt.figure(figsize=(12, 4))
plt.plot(freqs[mask], power[mask])
plt.xscale("log")
plt.yscale("log")
plt.title(f"Periodogram: {city_for_fft}")
plt.xlabel("Частота (1/час)")
plt.ylabel("Мощность")
plt.grid(True, alpha=0.3)
plt.show()

## 12. Feature engineering

In [ ]:
def add_calendar_features(df):
    df = df.copy()
    ts = pd.to_datetime(df[TIME_COL])

    df["hour"] = ts.dt.hour
    df["dayofweek"] = ts.dt.dayofweek
    df["dayofyear"] = ts.dt.dayofyear
    df["month"] = ts.dt.month
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

    # циклические признаки
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

    df["doy_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365.25)

    return df

feat_df = add_calendar_features(final_df)
display(feat_df.head())

In [ ]:
def make_origin_level_features(city_df, origin_idx, window=WINDOW, horizon=HORIZON):
    hist = city_df.iloc[origin_idx - window:origin_idx].copy()
    origin_row = city_df.iloc[origin_idx].copy()

    feats = {}

    feats["city"] = origin_row[CITY_COL]
    feats["origin_time"] = origin_row[TIME_COL]

    lag_list = [1, 2, 3, 6, 12, 24, 48, 72, 96, 120, 144, 168]
    for lag in lag_list:
        feats[f"temp_lag_{lag}"] = hist[TARGET_COL].iloc[-lag]

    for w in [6, 12, 24, 48, 72, 168]:
        feats[f"temp_mean_{w}"] = hist[TARGET_COL].tail(w).mean()
        feats[f"temp_std_{w}"] = hist[TARGET_COL].tail(w).std()
        feats[f"temp_min_{w}"] = hist[TARGET_COL].tail(w).min()
        feats[f"temp_max_{w}"] = hist[TARGET_COL].tail(w).max()

    other_cols = [
        "relative_humidity_2m", "precipitation", "rain", "snowfall",
        "weathercode", "wind_speed_10m", "surface_pressure"
    ]
    for col in other_cols:
        feats[f"{col}_last"] = hist[col].iloc[-1]
        feats[f"{col}_mean_24"] = hist[col].tail(24).mean()
        feats[f"{col}_mean_72"] = hist[col].tail(72).mean()

    feats["hour"] = origin_row["hour"]
    feats["dayofweek"] = origin_row["dayofweek"]
    feats["month"] = origin_row["month"]
    feats["is_weekend"] = origin_row["is_weekend"]
    feats["hour_sin"] = origin_row["hour_sin"]
    feats["hour_cos"] = origin_row["hour_cos"]
    feats["dow_sin"] = origin_row["dow_sin"]
    feats["dow_cos"] = origin_row["dow_cos"]
    feats["doy_sin"] = origin_row["doy_sin"]
    feats["doy_cos"] = origin_row["doy_cos"]

    y_seq = city_df[TARGET_COL].iloc[origin_idx:origin_idx + horizon].values

    return feats, y_seq

def build_direct_dataset(df, window=WINDOW, horizon=HORIZON, origin_stride=ORIGIN_STRIDE):
    rows = []
    ys = []

    for city, g in df.groupby(CITY_COL):
        g = g.sort_values(TIME_COL).reset_index(drop=True)
        n = len(g)

        for origin_idx in range(window, n - horizon + 1, origin_stride):
            feats, y_seq = make_origin_level_features(g, origin_idx, window=window, horizon=horizon)
            rows.append(feats)
            ys.append(y_seq)

    X = pd.DataFrame(rows)
    Y = np.vstack(ys)
    return X, Y

X_direct, Y_direct = build_direct_dataset(feat_df)
print(X_direct.shape, Y_direct.shape)
display(X_direct.head())

### Хронологическое разбиение train / val / test


In [ ]:
unique_origins = np.sort(X_direct["origin_time"].unique())

n_origins = len(unique_origins)
train_end = unique_origins[int(n_origins * 0.70)]
val_end = unique_origins[int(n_origins * 0.85)]

print("train_end:", train_end)
print("val_end:", val_end)

train_mask = X_direct["origin_time"] < train_end
val_mask = (X_direct["origin_time"] >= train_end) & (X_direct["origin_time"] < val_end)
test_mask = X_direct["origin_time"] >= val_end

X_train_direct = X_direct.loc[train_mask].reset_index(drop=True)
Y_train_direct = Y_direct[train_mask.values]

X_val_direct = X_direct.loc[val_mask].reset_index(drop=True)
Y_val_direct = Y_direct[val_mask.values]

X_test_direct = X_direct.loc[test_mask].reset_index(drop=True)
Y_test_direct = Y_direct[test_mask.values]

print(X_train_direct.shape, Y_train_direct.shape)
print(X_val_direct.shape, Y_val_direct.shape)
print(X_test_direct.shape, Y_test_direct.shape)

In [ ]:
FEATURE_DROP_COLS = ["origin_time"]
CAT_COLS = ["city"]
NUM_COLS = [c for c in X_train_direct.columns if c not in CAT_COLS + FEATURE_DROP_COLS]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), NUM_COLS),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore")),
        ]), CAT_COLS),
    ],
    remainder="drop"
)

X_train_direct_pre = preprocessor.fit_transform(X_train_direct)
X_val_direct_pre = preprocessor.transform(X_val_direct)
X_test_direct_pre = preprocessor.transform(X_test_direct)

print(type(X_train_direct_pre), X_train_direct_pre.shape)

## 13. Метрики качества прогноза


- **MAE** показывает среднюю абсолютную ошибку в тех же единицах, что и температура
  Это одна из самых понятных метрик: насколько в среднем модель ошибается по модулю

- **WAPE** показывает относительную ошибку по всему тестовому отрезку
  Она удобна тем, что учитывает масштаб ряда и позволяет сравнивать качество между разными сериями

- **MAPE** показывает среднюю относительную ошибку в процентах

- **Directional accuracy** показывает, насколько часто модель правильно угадывает направление изменения ряда:  
  выросла температура или снизилась

- **Directional R2** — дополнительная метрика, которая оценивает, насколько хорошо модель объясняет изменение направления и динамики ряда


In [ ]:
def wape(y_true, y_pred):
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(y_true - y_pred)) / denom

def mape_safe(y_true, y_pred, eps=1e-6):
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom))

def directional_accuracy(y_true, y_pred, last_observed):
    true_dir = np.sign(y_true - last_observed[:, None])
    pred_dir = np.sign(y_pred - last_observed[:, None])
    return np.mean(true_dir == pred_dir)

def directional_r2(y_true, y_pred, last_observed):
    true_delta = y_true - last_observed[:, None]
    pred_delta = y_pred - last_observed[:, None]
    ss_res = np.sum((true_delta - pred_delta) ** 2)
    ss_tot = np.sum((true_delta - np.mean(true_delta)) ** 2)
    if ss_tot == 0:
        return np.nan
    return 1 - ss_res / ss_tot

def evaluate_multihorizon(y_true, y_pred, last_observed):
    return {
        "WAPE": wape(y_true, y_pred),
        "MAE": np.mean(np.abs(y_true - y_pred)),
        "MAPE": mape_safe(y_true, y_pred),
        "directional_accuracy": directional_accuracy(y_true, y_pred, last_observed),
        "directional_r2": directional_r2(y_true, y_pred, last_observed),
    }

last_obs_test = X_test_direct["temp_lag_1"].values

## 14. Модели и подбор-гиперпараметров

In [ ]:
def make_model(model_name, params, strategy="direct"):
    if model_name == "dt":
        base = DecisionTreeRegressor(
            random_state=RANDOM_STATE,
            max_depth=params.get("max_depth"),
            min_samples_split=params.get("min_samples_split", 2),
            min_samples_leaf=params.get("min_samples_leaf", 1),
        )
        return base if strategy == "recursive" else MultiOutputRegressor(base)

    if model_name == "rf":
        base = RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_estimators=params.get("n_estimators", 200),
            max_depth=params.get("max_depth"),
            min_samples_split=params.get("min_samples_split", 2),
            min_samples_leaf=params.get("min_samples_leaf", 1),
            max_features=params.get("max_features", 1.0),
            n_jobs=N_JOBS_RF,
        )
        return base

    if model_name == "gbr":
        base = HistGradientBoostingRegressor(
            random_state=RANDOM_STATE,
            max_depth=params.get("max_depth"),
            learning_rate=params.get("learning_rate", 0.05),
            max_iter=params.get("max_iter", 300),
            min_samples_leaf=params.get("min_samples_leaf", 20),
            l2_regularization=params.get("l2_regularization", 0.0),
        )
        return base if strategy == "recursive" else MultiOutputRegressor(base)

    raise ValueError(model_name)

## Для direct-стратегии настраиваем гиперпараметры по подмножеству горизонтов

In [ ]:
VAL_HORIZON_SUBSET = [0, 23, 71, 167]

def direct_objective_factory(model_name):
    def objective(trial):
        if model_name == "dt":
            params = {
                "max_depth": trial.suggest_int("max_depth", 4, 20),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            }
        elif model_name == "rf":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 300),
                "max_depth": trial.suggest_int("max_depth", 6, 20),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
                "max_features": trial.suggest_float("max_features", 0.3, 1.0),
            }
        elif model_name == "gbr":
            params = {
                "max_depth": trial.suggest_int("max_depth", 3, 12),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "max_iter": trial.suggest_int("max_iter", 100, 350),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 80),
                "l2_regularization": trial.suggest_float("l2_regularization", 1e-6, 1.0, log=True),
            }
        else:
            raise ValueError(model_name)

        model = make_model(model_name, params, strategy="direct")
        model.fit(X_train_direct_pre, Y_train_direct[:, VAL_HORIZON_SUBSET])

        pred = model.predict(X_val_direct_pre)
        score = np.mean(np.abs(Y_val_direct[:, VAL_HORIZON_SUBSET] - pred))
        return score

    return objective

best_params_direct = {}
for model_name in ["dt", "rf", "gbr"]:
    print(f"\n=== Optuna direct: {model_name} ===")
    study = optuna.create_study(direction="minimize")
    study.optimize(direct_objective_factory(model_name), n_trials=N_TRIALS, show_progress_bar=False)
    best_params_direct[model_name] = study.best_params
    print("best value:", study.best_value)
    print("best params:", study.best_params)

best_params_direct

### По результатам подбора гиперпараметров для direct-стратегии лучшим оказался градиентный бустинг (gbr) с наименьшей валидационной ошибкой 2.278. На втором месте случайный лес (rf) с ошибкой 2.406, а дерево решений (dt) показало худший результат - 2.571. Это означает, что для прямого многошагового прогноза температуры бустинг лучше остальных моделей улавливает зависимость между признаками и будущими значениями температуры

## 15. Direct strategy: отдельная модель на каждый горизонт

In [ ]:
def fit_direct_per_horizon(model_name, params, X_train, Y_train):
    models = []
    for h in range(Y_train.shape[1]):
        model = make_model(model_name, params, strategy="recursive")
        model.fit(X_train, Y_train[:, h])
        models.append(model)
    return models

def predict_direct_per_horizon(models, X):
    preds = []
    for m in models:
        preds.append(m.predict(X))
    return np.column_stack(preds)

direct_test_results = {}
direct_models = {}

for model_name in ["dt", "rf", "gbr"]:
    print(f"\nFitting direct horizon-wise models: {model_name}")
    params = best_params_direct[model_name]
    models = fit_direct_per_horizon(model_name, params, X_train_direct_pre, Y_train_direct)
    pred_test = predict_direct_per_horizon(models, X_test_direct_pre)

    metrics = evaluate_multihorizon(Y_test_direct, pred_test, last_obs_test)
    direct_models[model_name] = models
    direct_test_results[model_name] = metrics
    print(metrics)

direct_test_df = pd.DataFrame(direct_test_results).T.sort_values("MAE")
display(direct_test_df)

## 16. Recursive strategy

Для recursive-стратегии обучаем одношаговую модель предсказания температуры на следующий час,  
потом раскатываем прогноз на 168 шагов вперёд

In [ ]:
def build_recursive_one_step_dataset(df, window=WINDOW):
    rows = []
    y = []

    for city, g in df.groupby(CITY_COL):
        g = g.sort_values(TIME_COL).reset_index(drop=True)
        n = len(g)

        for origin_idx in range(window, n - 1):
            hist = g.iloc[origin_idx - window:origin_idx].copy()
            origin_row = g.iloc[origin_idx].copy()

            feats = {"city": origin_row[CITY_COL], "origin_time": origin_row[TIME_COL]}

            lag_list = [1, 2, 3, 6, 12, 24, 48, 72, 96, 120, 144, 168]
            for lag in lag_list:
                feats[f"temp_lag_{lag}"] = hist[TARGET_COL].iloc[-lag]

            for w in [6, 12, 24, 48, 72, 168]:
                feats[f"temp_mean_{w}"] = hist[TARGET_COL].tail(w).mean()
                feats[f"temp_std_{w}"] = hist[TARGET_COL].tail(w).std()
                feats[f"temp_min_{w}"] = hist[TARGET_COL].tail(w).min()
                feats[f"temp_max_{w}"] = hist[TARGET_COL].tail(w).max()

            other_cols = [
                "relative_humidity_2m", "precipitation", "rain", "snowfall",
                "weathercode", "wind_speed_10m", "surface_pressure"
            ]
            for col in other_cols:
                feats[f"{col}_last"] = hist[col].iloc[-1]
                feats[f"{col}_mean_24"] = hist[col].tail(24).mean()
                feats[f"{col}_mean_72"] = hist[col].tail(72).mean()

            feats["hour"] = origin_row["hour"]
            feats["dayofweek"] = origin_row["dayofweek"]
            feats["month"] = origin_row["month"]
            feats["is_weekend"] = origin_row["is_weekend"]
            feats["hour_sin"] = origin_row["hour_sin"]
            feats["hour_cos"] = origin_row["hour_cos"]
            feats["dow_sin"] = origin_row["dow_sin"]
            feats["dow_cos"] = origin_row["dow_cos"]
            feats["doy_sin"] = origin_row["doy_sin"]
            feats["doy_cos"] = origin_row["doy_cos"]

            rows.append(feats)
            y.append(g[TARGET_COL].iloc[origin_idx])

    X = pd.DataFrame(rows)
    y = np.array(y)
    return X, y

X_rec, y_rec = build_recursive_one_step_dataset(feat_df)
print(X_rec.shape, y_rec.shape)
display(X_rec.head())

## Хронологическое разбиение recursive

In [44]:
unique_origins_rec = np.sort(X_rec["origin_time"].unique())
n_origins_rec = len(unique_origins_rec)

train_end_rec = unique_origins_rec[int(n_origins_rec * 0.70)]
val_end_rec = unique_origins_rec[int(n_origins_rec * 0.85)]

train_mask_rec = X_rec["origin_time"] < train_end_rec
val_mask_rec = (X_rec["origin_time"] >= train_end_rec) & (X_rec["origin_time"] < val_end_rec)
test_mask_rec = X_rec["origin_time"] >= val_end_rec

X_train_rec = X_rec.loc[train_mask_rec].reset_index(drop=True)
y_train_rec = y_rec[train_mask_rec.values]

X_val_rec = X_rec.loc[val_mask_rec].reset_index(drop=True)
y_val_rec = y_rec[val_mask_rec.values]

X_test_origins_rec = X_test_direct.copy()

NUM_COLS_REC = [c for c in X_train_rec.columns if c not in ["city", "origin_time"]]
CAT_COLS_REC = ["city"]

preprocessor_rec = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), NUM_COLS_REC),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore")),
        ]), CAT_COLS_REC),
    ],
    remainder="drop"
)

X_train_rec_pre = preprocessor_rec.fit_transform(X_train_rec)
X_val_rec_pre = preprocessor_rec.transform(X_val_rec)
print(X_train_rec_pre.shape, X_val_rec_pre.shape)

(257034, 73) (55080, 73)


In [ ]:
def recursive_objective_factory(model_name):
    def objective(trial):
        if model_name == "dt":
            params = {
                "max_depth": trial.suggest_int("max_depth", 4, 20),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            }
        elif model_name == "rf":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 300),
                "max_depth": trial.suggest_int("max_depth", 6, 20),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
                "max_features": trial.suggest_float("max_features", 0.3, 1.0),
            }
        elif model_name == "gbr":
            params = {
                "max_depth": trial.suggest_int("max_depth", 3, 12),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "max_iter": trial.suggest_int("max_iter", 100, 350),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 80),
                "l2_regularization": trial.suggest_float("l2_regularization", 1e-6, 1.0, log=True),
            }
        else:
            raise ValueError(model_name)

        model = make_model(model_name, params, strategy="recursive")
        model.fit(X_train_rec_pre, y_train_rec)
        pred = model.predict(X_val_rec_pre)
        score = mean_absolute_error(y_val_rec, pred)
        return score

    return objective

best_params_recursive = {}
for model_name in ["dt", "rf", "gbr"]:
    print(f"\n=== Optuna recursive: {model_name} ===")
    study = optuna.create_study(direction="minimize")
    study.optimize(recursive_objective_factory(model_name), n_trials=N_TRIALS, show_progress_bar=False)
    best_params_recursive[model_name] = study.best_params
    print("best value:", study.best_value)
    print("best params:", study.best_params)

best_params_recursive

[I 2026-05-10 12:45:14,739] A new study created in memory with name: no-name-52ba545c-e5ef-4c13-b763-04a34e8dede6



=== Optuna recursive: dt ===


[I 2026-05-10 12:45:36,290] Trial 0 finished with value: 0.7481688518056759 and parameters: {'max_depth': 17, 'min_samples_split': 30, 'min_samples_leaf': 12}. Best is trial 0 with value: 0.7481688518056759.
[I 2026-05-10 12:45:57,377] Trial 1 finished with value: 0.7466359335077405 and parameters: {'max_depth': 16, 'min_samples_split': 13, 'min_samples_leaf': 12}. Best is trial 1 with value: 0.7466359335077405.
[I 2026-05-10 12:46:06,139] Trial 2 finished with value: 0.914812777025903 and parameters: {'max_depth': 6, 'min_samples_split': 23, 'min_samples_leaf': 10}. Best is trial 1 with value: 0.7466359335077405.
[I 2026-05-10 12:46:25,880] Trial 3 finished with value: 0.7854332514549806 and parameters: {'max_depth': 15, 'min_samples_split': 7, 'min_samples_leaf': 4}. Best is trial 1 with value: 0.7466359335077405.
[I 2026-05-10 12:46:34,523] Trial 4 finished with value: 0.914812777025903 and parameters: {'max_depth': 6, 'min_samples_split': 16, 'min_samples_leaf': 10}. Best is trial 

best value: 0.728829261899065
best params: {'max_depth': 14, 'min_samples_split': 6, 'min_samples_leaf': 18}

=== Optuna recursive: rf ===


[I 2026-05-10 13:01:14,401] Trial 0 finished with value: 0.586367265784088 and parameters: {'n_estimators': 151, 'max_depth': 17, 'min_samples_split': 19, 'min_samples_leaf': 10, 'max_features': 0.4821572162750528}. Best is trial 0 with value: 0.586367265784088.


In [ ]:
city_groups = {
    city: feat_df[feat_df[CITY_COL] == city].sort_values(TIME_COL).reset_index(drop=True)
    for city in feat_df[CITY_COL].unique()
}

def update_recursive_feature_row(row, pred_history, future_ts):
    row = row.copy()

    series = np.array(pred_history)

    lag_list = [1, 2, 3, 6, 12, 24, 48, 72, 96, 120, 144, 168]
    for lag in lag_list:
        row[f"temp_lag_{lag}"] = series[-lag]

    for w in [6, 12, 24, 48, 72, 168]:
        tail = series[-w:]
        row[f"temp_mean_{w}"] = np.mean(tail)
        row[f"temp_std_{w}"] = np.std(tail)
        row[f"temp_min_{w}"] = np.min(tail)
        row[f"temp_max_{w}"] = np.max(tail)

    row["hour"] = future_ts.hour
    row["dayofweek"] = future_ts.dayofweek
    row["month"] = future_ts.month
    row["is_weekend"] = int(future_ts.dayofweek >= 5)
    row["hour_sin"] = np.sin(2 * np.pi * future_ts.hour / 24)
    row["hour_cos"] = np.cos(2 * np.pi * future_ts.hour / 24)
    row["dow_sin"] = np.sin(2 * np.pi * future_ts.dayofweek / 7)
    row["dow_cos"] = np.cos(2 * np.pi * future_ts.dayofweek / 7)
    row["doy_sin"] = np.sin(2 * np.pi * future_ts.dayofyear / 365.25)
    row["doy_cos"] = np.cos(2 * np.pi * future_ts.dayofyear / 365.25)

    return row

def recursive_forecast_for_origins(model, X_origins, preprocessor_rec, city_groups, horizon=HORIZON):
    preds = []

    for _, row in X_origins.iterrows():
        city = row["city"]
        origin_time = row["origin_time"]

        g = city_groups[city]
        origin_pos = g.index[g[TIME_COL] == origin_time][0]

        history = g[TARGET_COL].iloc[origin_pos-WINDOW:origin_pos].tolist()
        current_row = row.copy()

        local_preds = []
        for h in range(horizon):
            future_ts = origin_time + pd.Timedelta(hours=h)
            step_row = update_recursive_feature_row(current_row, history, future_ts)

            step_X = pd.DataFrame([step_row])
            step_X_pre = preprocessor_rec.transform(step_X)
            pred = model.predict(step_X_pre)[0]

            local_preds.append(pred)
            history.append(pred)

        preds.append(local_preds)

    return np.array(preds)

In [ ]:
recursive_test_results = {}
recursive_models = {}

for model_name in ["dt", "rf", "gbr"]:
    print(f"\nTraining recursive model: {model_name}")
    params = best_params_recursive[model_name]
    model = make_model(model_name, params, strategy="recursive")
    model.fit(X_train_rec_pre, y_train_rec)

    pred_test = recursive_forecast_for_origins(
        model=model,
        X_origins=X_test_origins_rec,
        preprocessor_rec=preprocessor_rec,
        city_groups=city_groups,
        horizon=HORIZON
    )

    metrics = evaluate_multihorizon(Y_test_direct, pred_test, last_obs_test)
    recursive_models[model_name] = model
    recursive_test_results[model_name] = metrics
    print(metrics)

recursive_test_df = pd.DataFrame(recursive_test_results).T.sort_values("MAE")
display(recursive_test_df)

## 17. Сравнение стратегий и моделей

In [ ]:
compare_df = (
    pd.concat({
        "direct": pd.DataFrame(direct_test_results).T,
        "recursive": pd.DataFrame(recursive_test_results).T,
    }, axis=0)
    .reset_index()
    .rename(columns={"level_0": "strategy", "level_1": "model"})
    .sort_values(["MAE", "WAPE"])
)

display(compare_df)

In [ ]:
plt.figure(figsize=(10, 4))
for strategy in compare_df["strategy"].unique():
    tmp = compare_df[compare_df["strategy"] == strategy]
    plt.plot(tmp["model"], tmp["MAE"], marker="o", label=strategy)

plt.title("Сравнение моделей по MAE")
plt.xlabel("Model")
plt.ylabel("MAE")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 18. Анализ остатков лучшей модели

In [ ]:
best_row = compare_df.sort_values("MAE").iloc[0]
best_strategy = best_row["strategy"]
best_model_name = best_row["model"]

print("Лучшая конфигурация:")
print(best_row)

if best_strategy == "direct":
    best_pred = predict_direct_per_horizon(direct_models[best_model_name], X_test_direct_pre)
else:
    best_pred = recursive_forecast_for_origins(
        model=recursive_models[best_model_name],
        X_origins=X_test_origins_rec,
        preprocessor_rec=preprocessor_rec,
        city_groups=city_groups,
        horizon=HORIZON
    )

residuals = (Y_test_direct - best_pred).ravel()

plt.figure(figsize=(12, 4))
plt.hist(residuals, bins=60)
plt.title(f"Распределение остатков: {best_strategy} / {best_model_name}")
plt.xlabel("Residual")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(residuals[:3000])
plt.title("Остатки на тесте (первые 3000 точек)")
plt.xlabel("Index")
plt.ylabel("Residual")
plt.grid(True, alpha=0.3)
plt.show()

## 19. Пример прогноза против факта

In [ ]:
sample_i = 0

plt.figure(figsize=(14, 4))
plt.plot(Y_test_direct[sample_i], label="fact")
plt.plot(best_pred[sample_i], label="forecast")
plt.title(f"Пример weekly-forecast: strategy={best_strategy}, model={best_model_name}")
plt.xlabel("Горизонт, часы")
plt.ylabel("Температура")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("Город:", X_test_direct.iloc[sample_i]['city'])
print("Origin:", X_test_direct.iloc[sample_i]['origin_time'])

## 20. Выводы

### Что было сделано
- Из Excel через `openpyxl` и `pandas` были извлечены все листы, включая скрытые
- Найдены и обработаны:
  - скрытые листы
  - битая кодировка
  - опечатки в названиях городов
  - строковые пропуски
  - смешанные типы
  - дубли по `(город, время)`
  - пропуски по временной оси
  - выбросы
- Построен единый аккуратный **panel dataframe** с часовым индексом
- Выполнены:
  - визуализация
  - ADF-проверка стационарности
  - STL-декомпозиция
  - спектральный анализ
- Созданы признаки:
  - лаги
  - календарные и циклические признаки
  - агрегаты по дополнительным метеопризнакам
- Обучены три семейства моделей:
  - дерево решений,
  - случайный лес,
  - бустинг.
- Реализованы и сравнены две стратегии:
  - recursive
  - direct
